In [6]:
from skillcorner.client import SkillcornerClient
from pprint import pprint
import pandas as pd
from io import BytesIO
from IPython.display import display
from skillcornerviz.standard_plots import summary_table as table
from skillcornerviz.utils import skillcorner_physical_utils as p_utils

In [7]:
import os
USERNAME = os.environ["SC_USERNAME"]
PASSWORD = os.environ["SC_PASSWORD"]
DATA_DIR = os.environ.get("DATA_DIR", os.path.expanduser("~/Desktop/analisis_ca/scout_dashboard/data"))

client = SkillcornerClient(username=USERNAME, password=PASSWORD)
client

In [8]:
# BAJAR DATOS PASES SOLO LIGA MX - COMPETITION EDITIONS 1169 + 1441

import os
import pandas as pd

COMPETITION_EDITIONS = [1169, 1441]

desktop_path = os.path.join(DATA_DIR, "passes")
os.makedirs(desktop_path, exist_ok=True)

average_per_configs = [
    ("match",                  "ligamx_per_match.csv"),
    ("100_pass_opportunities", "ligamx_per_100_pass_opportunities.csv"),
    ("30_min_tip",             "ligamx_per_30_min_tip.csv"),
]

for average_per, filename in average_per_configs:
    dfs = []

    for ce in COMPETITION_EDITIONS:
        data = client.get_in_possession_passes(
            params={
                "competition_edition": ce,
                "average_per": average_per
            }
        )
        df_ce = pd.DataFrame(data)
        df_ce["competition_edition"] = ce
        dfs.append(df_ce)
        print(f"  [{average_per}] Competition edition {ce}: {len(df_ce)} rows")

    df_combined = pd.concat(dfs, ignore_index=True)

    file_path = os.path.join(desktop_path, filename)
    df_combined.to_csv(file_path, index=False)
    print(f"Saved: {file_path} — Shape: {df_combined.shape}\n")

# BAJAR DATOS PASES SOLO LIGA MX

# import os
# import pandas as pd

# COMPETITION_EDITION = 1169

# # PER MATCH

# passes_match = client.get_in_possession_passes(
#     params={
#         "competition_edition": COMPETITION_EDITION,
#         "average_per": "match"
#     }
# )

# df_passes_match = pd.DataFrame(passes_match)

# desktop_path = os.path.join(DATA_DIR, "passes")
# os.makedirs(desktop_path, exist_ok=True)

# file_path = os.path.join(
#     desktop_path,
#     "ligamx_per_match.csv"
# )

# df_passes_match.to_csv(file_path, index=False)

# # PER 100 PASS OPPORTUNITIES

# passes_100_opps = client.get_in_possession_passes(
#     params={
#         "competition_edition": COMPETITION_EDITION,
#         "average_per": "100_pass_opportunities"
#     }
# )

# df_passes_100_opps = pd.DataFrame(passes_100_opps)

# desktop_path = os.path.join(DATA_DIR, "passes")
# os.makedirs(desktop_path, exist_ok=True)

# file_path = os.path.join(
#     desktop_path,
#     "ligamx_per_100_pass_opportunities.csv"
# )

# df_passes_100_opps.to_csv(file_path, index=False)

# # PER 30 MINUTES TIP

# passes_30_tip = client.get_in_possession_passes(
#     params={
#         "competition_edition": COMPETITION_EDITION,
#         "average_per": "30_min_tip"
#     }
# )

# df_passes_30_tip = pd.DataFrame(passes_30_tip)

# desktop_path = os.path.join(DATA_DIR, "passes")
# os.makedirs(desktop_path, exist_ok=True)

# file_path = os.path.join(
#     desktop_path,
#     "ligamx_per_30_min_tip.csv"
# )

# df_passes_30_tip.to_csv(file_path, index=False)

  [match] Competition edition 1169: 5216 rows
  [match] Competition edition 1441: 1304 rows
Saved: /Users/mateorodriguez/Desktop/analisis_ca/scout_dashboard/data/passes/ligamx_per_match.csv — Shape: (6520, 38)

  [100_pass_opportunities] Competition edition 1169: 5216 rows
  [100_pass_opportunities] Competition edition 1441: 1304 rows
Saved: /Users/mateorodriguez/Desktop/analisis_ca/scout_dashboard/data/passes/ligamx_per_100_pass_opportunities.csv — Shape: (6520, 38)

  [30_min_tip] Competition edition 1169: 5216 rows
  [30_min_tip] Competition edition 1441: 1304 rows
Saved: /Users/mateorodriguez/Desktop/analisis_ca/scout_dashboard/data/passes/ligamx_per_30_min_tip.csv — Shape: (6520, 38)



In [11]:

# DATOS DESMARQUES SOLO LIGA MX - COMPETITIONS 97 + 610

import os
import pandas as pd
from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_game_intelligence_utils as gi_utils

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

dfs = []

for comp_id in [97, 610]:
    params = {
        "season": 129,
        "competition": comp_id,
        "group_by": "player,team,competition,season,group",
        "playing_time__gte": 45,
        "count_match__gte": 5,
    }
    df_comp = pd.DataFrame(client.get_in_possession_off_ball_runs(params=params))
    dfs.append(df_comp)
    print(f"Competition {comp_id}: {len(df_comp)} rows")

df = pd.concat(dfs, ignore_index=True)

# Detect available name column
name_col = next((c for c in ["player_short_name", "short_name", "player_name"] if c in df.columns), None)
print("Using name column:", name_col)

# Aggregate to one row per player
meta_cols = ["player_id", "player_name", "team_id", "team_name", "season", "group", "competition"]
if name_col and name_col not in meta_cols:
    meta_cols.append(name_col)

sum_cols = [c for c in df.columns if c not in meta_cols and df[c].dtype in ['float64', 'int64']]

agg_kwargs = {
    "team_id": ("team_id", "last"),
    "team_name": ("team_name", "last"),
    **{col: (col, "sum") for col in sum_cols}
}

# Only include name column in agg if it exists and isn't a groupby key
if name_col and name_col != "player_name":
    agg_kwargs[name_col] = (name_col, "last")

df_agg = df.groupby(["player_id", "player_name"], as_index=False).agg(**agg_kwargs)

# Re-normalize on combined totals
gi_utils.add_run_normalisations(df_agg)

output_dir = os.path.join(DATA_DIR, "off_ball_runs")
os.makedirs(output_dir, exist_ok=True)

file_path = os.path.join(output_dir, "ligamx_obr_normalized.csv")
df_agg.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Shape:", df_agg.shape)

# # DATOS DESMARQUES SOLO LIGA MX

# import os
# import pandas as pd
# from skillcorner.client import SkillcornerClient
# from skillcornerviz.utils import skillcorner_game_intelligence_utils as gi_utils

# client = SkillcornerClient(username=USERNAME, password=PASSWORD)

# params = {
#     "season": 129,
#     "competition": 97,
#     "group_by": "player,team,competition,season,group",
#     "playing_time__gte": 45,
#     "count_match__gte": 5,
# }

# df = pd.DataFrame(
#     client.get_in_possession_off_ball_runs(params=params)
# )

# gi_utils.add_run_normalisations(df)

# output_dir = os.path.join(DATA_DIR, "off_ball_runs")
# os.makedirs(output_dir, exist_ok=True)

# file_path = os.path.join(output_dir, "ligamx_obr_normalized.csv")
# df.to_csv(file_path, index=False)


Competition 97: 287 rows
Competition 610: 45 rows
Using name column: short_name
Saved to: /Users/mateorodriguez/Desktop/analisis_ca/scout_dashboard/data/off_ball_runs/ligamx_obr_normalized.csv
Shape: (271, 52)


In [10]:
# DATOS PRESIONES SOLO LIGA MX - COMPETITION EDITIONS 1169 + 1441

import os
import pandas as pd
from functools import reduce

COMPETITION_EDITIONS = [1169, 1441]

JOIN_KEYS = [
    "player_id", "match_id", "player_name", "player_birthdate",
    "match_name", "match_date",
    "team_id", "team_name",
    "competition_id", "competition_name",
    "season_id", "season_name",
    "competition_edition_id",
    "position", "group", "short_name",
    "result", "venue",
    "third", "channel",
    "minutes_played_per_match",
    "adjusted_min_tip_per_match",
    "quality_check"
]

AVERAGE_PERS = {
    "match": "pm",
    "100_pressures": "p100press",
    "30_min_tip": "p30tip"
}

def fetch_pressures(avg: str, competition_edition: int) -> pd.DataFrame:
    data = client.get_in_possession_on_ball_pressures(
        params={
            "competition_edition": competition_edition,
            "average_per": avg
        }
    )
    df = pd.DataFrame(data)

    missing = [k for k in JOIN_KEYS if k not in df.columns]
    if missing:
        raise KeyError(
            f"Missing join keys {missing} for average_per='{avg}'. "
            f"Available columns: {list(df.columns)[:80]} ..."
        )
    return df

def suffix_non_keys(df: pd.DataFrame, suffix: str) -> pd.DataFrame:
    rename_map = {c: f"{c}_{suffix}" for c in df.columns if c not in JOIN_KEYS}
    return df.rename(columns=rename_map)

# Fetch, merge wide per edition, then concatenate
all_editions = []

for ce in COMPETITION_EDITIONS:
    print(f"Fetching competition edition {ce}...")
    dfs = [suffix_non_keys(fetch_pressures(avg, ce), suf) for avg, suf in AVERAGE_PERS.items()]

    df_wide_ce = reduce(
        lambda l, r: l.merge(r, on=JOIN_KEYS, how="outer", validate="one_to_one"),
        dfs
    )
    df_wide_ce["competition_edition"] = ce
    all_editions.append(df_wide_ce)
    print(f"  Competition edition {ce}: {df_wide_ce.shape[0]} rows")

df_wide = pd.concat(all_editions, ignore_index=True)

# Reorder: keys first
metric_cols = [c for c in df_wide.columns if c not in JOIN_KEYS + ["competition_edition"]]
df_wide = df_wide[JOIN_KEYS + ["competition_edition"] + metric_cols]

# Save
desktop_path = os.path.join(DATA_DIR, "pressures")
os.makedirs(desktop_path, exist_ok=True)

file_path = os.path.join(desktop_path, "ligamx_on_ball_pressures.csv")
df_wide.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Shape:", df_wide.shape)

# # DATOS PRESIONES SOLO LIGA MX

# from functools import reduce

# COMPETITION_EDITION = 1169

# JOIN_KEYS = [
#     "player_id", "match_id", "player_name", "player_birthdate",
#     "match_name", "match_date",
#     "team_id", "team_name",
#     "competition_id", "competition_name",
#     "season_id", "season_name",
#     "competition_edition_id",
#     "position", "group", "short_name",
#     "result", "venue",
#     "third", "channel",
#     "minutes_played_per_match",
#     "adjusted_min_tip_per_match",
#     "quality_check"
# ]

# # ✅ Correct average_per values for on_ball_pressures
# AVERAGE_PERS = {
#     "match": "pm",
#     "100_pressures": "p100press",
#     "30_min_tip": "p30tip"
# }

# def fetch_pressures(avg: str) -> pd.DataFrame:
#     data = client.get_in_possession_on_ball_pressures(
#         params={
#             "competition_edition": COMPETITION_EDITION,
#             "average_per": avg
#         }
#     )
#     df = pd.DataFrame(data)

#     missing = [k for k in JOIN_KEYS if k not in df.columns]
#     if missing:
#         raise KeyError(
#             f"Missing join keys {missing} for average_per='{avg}'. "
#             f"Available columns: {list(df.columns)[:80]} ..."
#         )
#     return df

# def suffix_non_keys(df: pd.DataFrame, suffix: str) -> pd.DataFrame:
#     rename_map = {c: f"{c}_{suffix}" for c in df.columns if c not in JOIN_KEYS}
#     return df.rename(columns=rename_map)

# # Fetch + suffix
# dfs = [suffix_non_keys(fetch_pressures(avg), suf) for avg, suf in AVERAGE_PERS.items()]

# # Merge wide
# df_wide = reduce(
#     lambda l, r: l.merge(r, on=JOIN_KEYS, how="outer", validate="one_to_one"),
#     dfs
# )

# # Reorder: keys first
# metric_cols = [c for c in df_wide.columns if c not in JOIN_KEYS]
# df_wide = df_wide[JOIN_KEYS + metric_cols]

# # Save
# desktop_path = os.path.join(DATA_DIR, "pressures")
# os.makedirs(desktop_path, exist_ok=True)

# file_path = os.path.join(desktop_path, "ligamx_on_ball_pressures.csv")
# df_wide.to_csv(file_path, index=False)


Fetching competition edition 1169...
  Competition edition 1169: 5216 rows
Fetching competition edition 1441...
  Competition edition 1441: 1304 rows
Saved to: /Users/mateorodriguez/Desktop/analisis_ca/scout_dashboard/data/pressures/ligamx_on_ball_pressures.csv
Shape: (6520, 66)
